<a href="https://colab.research.google.com/github/ZiyuCao421/26-CP5046-SIN-CAM-TR3-ICT-Project-1-Analysis-and-Design/blob/main/V3/notebooks/01_all_feature_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V3 — Stage 1: All-Candidate Single-Feature Screening

**Purpose.** This notebook is the first stage of the V3 workflow. It builds a candidate feature
pool from the course mouse-dynamics CSV files and evaluates **every candidate feature on its own**
under one fixed protocol, so that later stages can choose feature sets on evidence rather than on
assumption.

**Stage boundary.** This notebook performs *single-feature screening only*. It does not select, tune
or evaluate any multi-feature subset. The stage-2 feature-set notebook is deliberately **not**
created here and must not be started until this stage has been executed on the real data and
reviewed.

> **Results validity.** Every table, file and number this notebook can produce is
> **only valid after a real Colab run on the real course data**. This notebook is committed
> **without stored outputs** and contains **no result values of any kind**. Any metric read from it
> before such a run is fabricated by definition.

## How the comparison is made fair

A feature can only be ranked against another feature if both were measured on the same data. Two
traps make a naive per-feature screen invalid, even with a fixed random seed:

* **Different rows.** Each feature has its own missing and infinite values, so a per-feature
  screen would build a different sample for each column, and the train/test rows would differ.
* **Different classes.** A per-feature screen also drops rare labels per feature, so the class
  space would differ from one feature to the next.
* **A shared seed is not a shared split.** `train_test_split` with the same `random_state` on
  *different* row sets produces *different* splits. An identical seed guarantees reproducibility,
  not comparability.

This notebook therefore builds **one global primary cohort** and draws **one single stratified
split** on it:

1. All matched CSV files are merged first, because a single participant file usually carries a
   single label; the class space only exists across the merged data.
2. The cohort keeps the rows whose label is present, whose label class has at least
   `MIN_CLASS_COUNT` rows, and for which **every** candidate column is finite after numeric
   coercion (missing and ±inf both count as missing). The number of removed rows and the reason
   for each removal are recorded.
3. **One** stratified `train_test_split` is drawn on that cohort, and its fingerprint is recorded.
4. Every candidate feature is then evaluated on exactly the same training rows, the same test rows
   and the same class space.

Per-feature data-quality diagnostics (non-null rate, coercion rate, infinite count, constant
columns) are still reported, but they never rebuild a sample and never change the cohort.

An alternative `per_feature` mode exists for inspection only. It is stamped
`exploratory / not rankable`, is not the default, and must not be used for stage-2 selection.

**Data privacy.** The course data is behavioural biometric data (mouse dynamics) carrying
per-participant labels. This notebook:

* reads the CSV files from a directory that you configure, and never writes, uploads, copies or
  transmits them;
* never commits raw rows, raw columns or participant-linked file names to GitHub;
* writes only **derived, aggregated** artefacts (schema inventory, exclusion reasons, per-feature
  metrics, run manifest);
* never records a local absolute path in the manifest;
* **refuses** to write its output tables into a directory that belongs to a Git repository
  (see `ALLOW_OUTPUT_IN_GIT_REPO` in the configuration cell).

**What the stage produces.** Five files, written to `OUTPUT_DIR`:

| File | Content |
| --- | --- |
| `feature_inventory.csv` | Every column of the merged data, its type, its screening status and the rule that decided it |
| `excluded_features.csv` | Only the rejected columns, with the exact exclusion reason and the matched rule |
| `single_feature_metrics.csv` | One row per (candidate feature × model) with the held-out metrics and the shared split |
| `single_feature_failures.csv` | Every feature or model that could not be evaluated, with the reason |
| `run_metadata.json` | The manifest: cohort definition and accounting, split fingerprint, configuration snapshot, environment, data footprint, validity stamp |

**Integrity rules enforced by the code.**

1. The label is read **from inside the CSV files only**. It is never derived from a file name, never
   renamed and never overwritten.
2. The cohort and the split are built **once** and shared by every candidate feature.
3. The `StandardScaler` for the SVC lives inside a `Pipeline` and is therefore fitted on training
   data only.
4. No hyper-parameter search is performed, and the test portion is never used to select anything.
5. A failing column or model is recorded and the run continues with the next one.


## How to run this notebook in Google Colab

1. **Open the notebook in Colab** and set *Runtime → Run all*. The runtime must be able to import
   `numpy`, `pandas` and `scikit-learn`; Colab's standard image already provides all three.
2. **Mount Drive** (optional but usual for this dataset) — run the following in a cell *before* the
   configuration cell, or let the discovery cell tell you what is missing:

   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```

3. **Edit the single configuration cell** (`02_config.py` in the source layout, the cell headed
   *SINGLE EDITABLE CONFIGURATION BLOCK*). Set at least `DATA_DIR` to the folder that holds the
   course CSV files. Everything else has a working default.
4. **Run all cells** in order. Each cell is defensive: a missing directory, a missing label column,
   a dirty numeric column or a model that refuses to fit is reported and the run continues.
5. **Read the outputs** from `OUTPUT_DIR` (a path printed in the output cell). Copy the five
   aggregated files into `V3/results/single_feature/` in the repository **only if you have decided
   that to do so**, and **never** copy any raw data or participant-linked file names. The repository
   should also ignore `V3/results/**` data files before you commit.
6. **Record the run.** `run_metadata.json` contains the configuration snapshot, the environment and
   the data footprint. Quote that manifest — not a copied table cell — when reporting results.

### If the data folder is a shared Drive folder

Add the shared folder to *My Drive* (right-click → *Add shortcut to My Drive*) and point `DATA_DIR`
at the shortcut path, for example `/content/drive/MyDrive/<folder name>`. Do not point `DATA_DIR` at
a read-only mount if you intend to write outputs there — write outputs to a separate directory.

### What this notebook will not do

* It will not run without `DATA_DIR` pointing at real files: there is no bundled sample, no
  synthetic fallback and no cached result.
* It will not write anything into a Git repository.
* It will not derive a label from a file name if the label column is missing; the affected file is
  skipped and reported instead.


In [6]:
# =============================================================================
#  ENVIRONMENT
#  Nothing here is editable. It only reports the runtime the results came from.
# =============================================================================
import hashlib
import json
import platform
import re
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=FutureWarning)

print("python      :", sys.version.split()[0])
print("platform    :", platform.platform())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)


python      : 3.13.16
platform    : Linux-6.6.159+-x86_64-with-glibc2.39
numpy       : 2.4.6
pandas      : 2.2.3
scikit-learn: 1.6.1


In [7]:
# =============================================================================
#  SINGLE EDITABLE CONFIGURATION BLOCK
#  ---------------------------------------------------------------------------
#  Edit ONLY this cell. Every other cell reads these names and nothing else.
#  A mirror of this block (with descriptions) is committed at
#  V3/configs/screening_config.json for review and for the run manifest.
# =============================================================================

# --- Data location ----------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics"  # folder holding the course CSV files
FILE_PATTERN = "*.csv"          # glob for the per-participant CSV files
LABEL_COLUMN = "label"          # the label column exactly as it appears INSIDE the CSV files

# --- Reproducibility --------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20                # held-out fraction of the single shared split on the common cohort

# --- Screening cohort -------------------------------------------------------
# "complete_case_all_candidates" : PRIMARY. One global complete-case cohort is built
#     over all candidate columns and one single stratified split is drawn on it.
#     Every candidate feature is then evaluated on exactly the same training rows,
#     the same test rows and the same class space. This is the only rankable mode.
# "per_feature" : EXPLORATORY ONLY. Rebuilds a sample and a split for each feature.
#     The samples differ between features, so the result cannot be ranked and must
#     not be used for stage-2 selection. Never the default.
SCREENING_COHORT_MODE = "complete_case_all_candidates"

# --- Data-quality thresholds ------------------------------------------------
MIN_NON_MISSING_RATE = 0.90     # a candidate must have at least this share of usable values
MIN_CLASS_COUNT = 5             # classes with fewer rows than this are dropped before splitting
MIN_ROWS_PER_FEATURE = 100      # a feature needs at least this many usable rows to be screened
MIN_COERCION_RATE = 0.95        # share of non-null values of a text column that must convert to float

# --- Output -----------------------------------------------------------------
OUTPUT_DIR = "/content/outputs/v3_single_feature"   # aggregated tables only, never raw data

# --- Explicit exclusions ----------------------------------------------------
EXCLUDED_COLUMNS = [
    "Unnamed: 0", "Unnamed: 1", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4",
    "index", "Index", "level_0",
    "Seg_id",      # per-segment identifier, not a behaviour measure
]

# --- Protected feature families (never removed by a pattern rule) ------------
# These are the behavioural and statistical measures of the dataset. They are
# checked BEFORE the identifier / coordinate / timestamp rules, so that an
# uppercase name or a numeric suffix can never exclude them by accident.
PROTECTED_COLUMN_PATTERNS = [
    r"^avg_[A-Za-z0-9_]+$",               # avg_* summary measures
    r"^min_[A-Za-z0-9_]+$",               # min_* summary measures
    r"^max_[A-Za-z0-9_]+$",               # max_* summary measures
    r"^(d|TV|HV|VV|A|AM|AV|J)\d{0,2}$",   # per-point families: d0-d2, TV0-TV2, HV0-HV2,
                                           # VV0-VV2, A0-A2, AM0-AM2, AV0-AV2, J0-J2
]

# --- Rule-based exclusions (auditable, configurable) ------------------------
EXCLUDED_COLUMN_PATTERNS = [
    r"^unnamed",            # pandas export columns
    r"^__index_level",      # multi-index export columns
]
IDENTIFIER_COLUMN_PATTERNS = [
    r"(^|_)(id|uid|pid|guid)$",
    r"(^|_)(user|participant|subject|person|student|patient)s?(_|$)",
    r"(^|_)(file|filename|source|src|origin|dataset|filepath|folder|path)s?(_|$)",
    r"(^|_)(session|recording|clip|run|trial|device|machine|site|lab)($|_)",
]
RAW_COORDINATE_PATTERNS = [
    r"^(x|xs|xx|y|ys|yy|pos|position|coord|coords|px|py)(_?\d*)$",   # x0-x2, y0-y2 and the like
]
TIMESTAMP_PATTERNS = [
    r"^(t|ts|time|timestamp|datetime|date|epoch|unix_time)$",
    r"^t\d+$",                                                     # t0-t2 raw times
    r"(^|_)(timestamp|datetime|epoch|unix|utc|local_time|collect(ed|ion)?_time)(_|$)",
]

# --- Safety switches --------------------------------------------------------
ALLOW_OUTPUT_IN_GIT_REPO = False   # refuse to write outputs into a git checkout
INCLUDE_SOURCE_FILE_NAMES = False  # keep participant-linked file names out of the outputs
INCLUDE_LABEL_VALUES = False       # keep participant-linked label values out of the outputs


## Configuration reference

Every knob lives in the single editable cell above. Nothing else in the notebook needs to be
changed for a normal run.

| Name | Meaning | Effect when changed |
| --- | --- | --- |
| `DATA_DIR` | Folder that holds the course CSV files | The only setting you normally must edit. |
| `FILE_PATTERN` | Glob used to select the files | Use e.g. `"*_Segment.csv"` to narrow the set. |
| `LABEL_COLUMN` | Name of the label column **inside** the CSVs | Files without this column are skipped and reported; the label is never inferred from a file name. |
| `RANDOM_STATE` | Seed for the split and for the models | The same seed reproduces the same split and the same forest. |
| `TEST_SIZE` | Fraction held out for evaluation | Applied to the single shared split. |
| `SCREENING_COHORT_MODE` | `"complete_case_all_candidates"` (default) or `"per_feature"` | The default builds one global cohort and one shared split, so features are comparable. `"per_feature"` is exploratory, is stamped *not rankable*, and must not drive stage-2 selection. |
| `MIN_NON_MISSING_RATE` | Minimum share of usable (present, finite, numeric-coercible) values | A column below the threshold is excluded with reason `insufficient_non_missing_rate`. |
| `MIN_CLASS_COUNT` | Minimum number of rows per label | Classes below the threshold are dropped before the cohort is built; the affected rows are counted, not published. |
| `MIN_ROWS_PER_FEATURE` | Minimum usable rows before a candidate is screened | Below it, the feature is recorded in the failure table. |
| `MIN_COERCION_RATE` | Share of non-null values of a text column that must convert to float | Below it, the column is excluded as `not_numeric_or_safely_coercible`. |
| `OUTPUT_DIR` | Where the five aggregated files are written | A Git checkout is refused as a target unless `ALLOW_OUTPUT_IN_GIT_REPO` is set to `True`. |
| `EXCLUDED_COLUMNS` | Exact column names to drop | Compared case-insensitively. Holds the blank-header index column and `Seg_id`. |
| `EXCLUDED_COLUMN_PATTERNS` | Regular expressions for columns to drop | Applied case-insensitively. |
| `PROTECTED_COLUMN_PATTERNS` | Families that the **name** rules may never remove | Covers `avg_*`, `min_*`, `max_*` and the per-point families `d`, `TV`, `HV`, `VV`, `A`, `AM`, `AV`, `J` with an optional numeric suffix. Checked **before** the identifier / coordinate / timestamp rules, so an uppercase name or a numeric suffix can never exclude a behavioural measure by accident. The type, constant and completeness tests still apply to a protected column. |
| `IDENTIFIER_COLUMN_PATTERNS` | Regular expressions catching user / file / source identifiers | The matched pattern is written to the exclusion table. |
| `RAW_COORDINATE_PATTERNS` | Regular expressions catching raw screen coordinates | Removes `x0`–`x2`, `y0`–`y2` and the like. |
| `TIMESTAMP_PATTERNS` | Regular expressions catching raw time values | Removes `t0`–`t2`, `t`, `timestamp`, `datetime` and the like. |
| `ALLOW_OUTPUT_IN_GIT_REPO` | Safety switch | Keep `False`. Set to `True` only if a deliberate decision is made to keep results inside the repository. |
| `INCLUDE_SOURCE_FILE_NAMES` | Whether the output manifest lists source file names | Keep `False`: the file names are participant-linked. |
| `INCLUDE_LABEL_VALUES` | Whether label values appear in the outputs | Keep `False`: the labels identify participants. |

**The exclusion rules are the audit trail.** Whatever is removed from the candidate pool, and for
which reason, is written to `excluded_features.csv` with the rule that decided it. A column is
never dropped silently, and a protected family is never removed by a name rule.

**Committed mirror.** `V3/configs/screening_config.json` holds the same values together with a
one-line reason for each threshold. If the two ever disagree, the notebook cell is the authority
for the run and the JSON file must be updated in the same commit.


In [8]:
# =============================================================================
#  HELPERS  (not editable)
# =============================================================================
RESULTS_VALIDITY = "only valid after a real Colab run"

# Columns that are bookkeeping for this notebook and can never become features.
SOURCE_FILE_COLUMN = "__source_file__"


def utc_now_iso():
    """Timestamp used in the manifest and in failure rows."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_text(text):
    """Stable digest used to fingerprint the input file set without naming it."""
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def find_git_root(path):
    """Return the nearest ancestor directory that contains a .git entry, else None."""
    p = Path(path).expanduser().resolve()
    if p.is_file():
        p = p.parent
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    return None


def ensure_output_dir(output_dir):
    """Create the output directory, refusing a Git checkout as the target."""
    out = Path(output_dir).expanduser()
    if not ALLOW_OUTPUT_IN_GIT_REPO:
        git_root = find_git_root(out)
        if git_root is not None:
            raise RuntimeError(
                "Refusing to write screening outputs inside a Git repository "
                f"({git_root}). Course-derived tables must not be committed. "
                "Choose an OUTPUT_DIR outside the repository, or set "
                "ALLOW_OUTPUT_IN_GIT_REPO = True deliberately."
            )
    out.mkdir(parents=True, exist_ok=True)
    return out


def match_pattern(name, patterns):
    """Return the first pattern that matches the column name, else None."""
    for pattern in patterns:
        if re.search(pattern, str(name), flags=re.IGNORECASE):
            return pattern
    return None


def exact_match(name, names):
    """Case-insensitive membership test over a list of exact names."""
    lowered = {str(n).lower() for n in names}
    return str(name).lower() in lowered


def dump_json(path, payload):
    """Write a JSON file with a stable, readable layout."""
    Path(path).write_text(
        json.dumps(payload, indent=2, default=str, ensure_ascii=False),
        encoding="utf-8",
    )
    return path


def write_table(frame, path, validity=RESULTS_VALIDITY):
    """Write a derived table with the validity stamp attached to every row."""
    out = frame.copy()
    out["results_validity"] = validity
    out.to_csv(path, index=False)
    return path


print("helpers loaded; validity stamp =", repr(RESULTS_VALIDITY))


helpers loaded; validity stamp = 'only valid after a real Colab run'


In [9]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# =============================================================================
#  STEP 1 — DISCOVER THE INPUT FILES
#  Lists the files that match FILE_PATTERN. Reads nothing yet.
# =============================================================================
data_dir_path = Path(DATA_DIR).expanduser()

if not data_dir_path.exists():
    raise FileNotFoundError(
        f"DATA_DIR does not exist: {data_dir_path}\n"
        "Mount Google Drive (from google.colab import drive; drive.mount('/content/drive')) "
        "and set DATA_DIR to the folder that holds the course CSV files."
    )
if not data_dir_path.is_dir():
    raise NotADirectoryError(f"DATA_DIR is not a directory: {data_dir_path}")

source_files = sorted(p for p in data_dir_path.glob(FILE_PATTERN) if p.is_file())

if not source_files:
    raise FileNotFoundError(
        f"No files matching {FILE_PATTERN!r} were found in {data_dir_path}. "
        "Check DATA_DIR and FILE_PATTERN. The notebook has no bundled sample data."
    )

print("Data directory: <configured; path withheld from notebook output>")
print(f"FILE_PATTERN : {FILE_PATTERN}")
print(f"Files found  : {len(source_files)}")
print()
print("Only the aggregate file count is printed. No file names or file content are displayed.")

# Fail fast if the output target is not acceptable, before any expensive work.
output_dir_path = ensure_output_dir(OUTPUT_DIR)
print(f"Output directory ready: {output_dir_path}")


Data directory: <configured; path withheld from notebook output>
FILE_PATTERN : *.csv
Files found  : 10

Only the aggregate file count is printed. No file names or file content are displayed.
Output directory ready: /content/outputs/v3_single_feature


In [11]:
# =============================================================================
#  STEP 2 — LOAD AND MERGE
#  * The label is taken from inside the CSV files and is never derived from a
#    file name, renamed or overwritten.
#  * A file without the label column is skipped and reported.
#  * A file whose columns cannot be typed automatically is re-read as text and
#    coerced later, per feature.
#  * No raw value is printed and nothing is written to disk at this stage.
# =============================================================================
frames = []
file_records = []
skipped_files = []

for path in source_files:
    try:
        chunk = pd.read_csv(path)
    except Exception as exc:  # dirty numeric column, stray quote, ...
        try:
            chunk = pd.read_csv(path, dtype=str)
            read_note = f"re-read as text ({type(exc).__name__})"
        except Exception as exc2:
            skipped_files.append(
                {"file": path.name, "reason": f"unreadable_csv: {type(exc2).__name__}"}
            )
            continue
    else:
        read_note = "read as typed"

    if LABEL_COLUMN not in chunk.columns:
        skipped_files.append(
            {"file": path.name, "reason": f"missing label column {LABEL_COLUMN!r}"}
        )
        continue

    chunk = chunk.copy()
    # Bookkeeping only: keeps the origin of every row for auditing.
    # It is excluded from the candidate pool by the reserved-column rule.
    chunk[SOURCE_FILE_COLUMN] = path.name
    frames.append(chunk)
    file_records.append(
        {
            "file": path.name,
            "n_rows": int(len(chunk)),
            "n_columns": int(len(chunk.columns)),
            "read_note": read_note,
        }
    )

if not frames:
    raise RuntimeError(
        "No usable file was loaded. Every matched file was skipped. "
        f"Reasons: {skipped_files}. The label is never derived from a file name, "
        "so a file without the label column cannot contribute rows."
    )

df = pd.concat(frames, ignore_index=True, sort=False)

print(f"Files loaded : {len(frames)}")
print(f"Files skipped: {len(skipped_files)}")
for record in skipped_files:
    print(f"  skipped: {record['file']} -> {record['reason']}")
print(f"Merged shape : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Label column : {LABEL_COLUMN!r} present = {LABEL_COLUMN in df.columns}")
print()

if LABEL_COLUMN not in df.columns:  # defensive; cannot happen after the checks above
    raise RuntimeError(f"Label column {LABEL_COLUMN!r} is missing from the merged frame.")

n_label_missing = int(df[LABEL_COLUMN].isna().sum())
label_counts = df[LABEL_COLUMN].value_counts(dropna=True)
print(f"Rows without a label : {n_label_missing}")
print(f"Distinct label values: {len(label_counts)}")
if INCLUDE_LABEL_VALUES:
    print(label_counts.to_string())
else:
    counts = label_counts.to_numpy()
    print(
        "Per-label counts withheld (INCLUDE_LABEL_VALUES=False). "
        f"min={counts.min()}, median={int(np.median(counts))}, max={counts.max()}"
    )
print()
print("No raw rows are displayed. The merged frame stays in memory and is never written to disk.")


Files loaded : 10
Files skipped: 0
Merged shape : 2002 rows x 70 columns
Label column : 'label' present = True

Rows without a label : 0
Distinct label values: 9
Per-label counts withheld (INCLUDE_LABEL_VALUES=False). min=122, median=211, max=365

No raw rows are displayed. The merged frame stays in memory and is never written to disk.


In [12]:
# =============================================================================
#  STEP 3 — BUILD THE CANDIDATE FEATURE POOL
#  Every column of the merged frame is classified, and the reason for every
#  rejection is recorded. The rules are the ones declared in the configuration
#  cell, so the pool is reproducible and auditable.
#
#  Rule order is fixed. PROTECTED_COLUMN_PATTERNS are checked BEFORE the
#  identifier / coordinate / timestamp rules, so that a behavioural measure
#  (avg_*, min_*, max_*, d0-d2, TV0-TV2, HV0-HV2, VV0-VV2, A0-A2, AM0-AM2,
#  AV0-AV2, J0-J2) can never be removed by an uppercase name or a numeric
#  suffix. The protection only overrides the NAME rules; the type, constant and
#  completeness tests still apply to a protected column.
# =============================================================================
RESERVED_NON_FEATURE_COLUMNS = {LABEL_COLUMN, SOURCE_FILE_COLUMN}

inventory_rows = []
excluded_rows = []
candidate_features = []

for column in df.columns:
    series = df[column]
    n_rows = int(len(df))
    n_non_null = int(series.notna().sum())
    n_unique_raw = int(series.nunique(dropna=True))
    is_numeric_dtype = bool(pd.api.types.is_numeric_dtype(series))

    # --- how far can this column be used as a number? -----------------------
    coercion_ok = True
    try:
        coerced = pd.to_numeric(series, errors="coerce")
        n_coerced = int(coerced.notna().sum())
        n_infinite = int(
            np.isinf(coerced.to_numpy(dtype="float64", na_value=np.nan)).sum()
        )
        n_unique_coerced = int(coerced.nunique(dropna=True))
    except (TypeError, ValueError):
        coercion_ok = False
        coerced = None
        n_coerced = 0
        n_infinite = 0
        n_unique_coerced = 0

    n_coercion_failures = max(0, n_non_null - n_coerced)
    coercion_rate = (n_coerced / n_non_null) if n_non_null else 0.0
    n_usable = max(0, n_coerced - n_infinite)
    usable_rate = (n_usable / n_rows) if n_rows else 0.0

    # --- apply the exclusion rules, in a fixed order ------------------------
    reason = None
    matched_rule = None
    detail = ""
    protected = False

    if column in RESERVED_NON_FEATURE_COLUMNS:
        reason = (
            "target_label_column"
            if column == LABEL_COLUMN
            else "source_file_bookkeeping_column"
        )
        detail = "reserved by the protocol, never a candidate feature"
    elif exact_match(column, EXCLUDED_COLUMNS):
        reason = "config_excluded_column"
        detail = "listed in EXCLUDED_COLUMNS"
    elif match_pattern(column, EXCLUDED_COLUMN_PATTERNS) is not None:
        reason = "config_excluded_pattern"
        matched_rule = match_pattern(column, EXCLUDED_COLUMN_PATTERNS)
        detail = f"matched EXCLUDED_COLUMN_PATTERNS {matched_rule!r}"
    else:
        protected_rule = match_pattern(column, PROTECTED_COLUMN_PATTERNS)
        protected = protected_rule is not None
        identifier_rule = match_pattern(column, IDENTIFIER_COLUMN_PATTERNS)
        coordinate_rule = match_pattern(column, RAW_COORDINATE_PATTERNS)
        timestamp_rule = match_pattern(column, TIMESTAMP_PATTERNS)

        if protected:
            # A protected family is never removed by a name rule. It is still
            # subject to the type, constant and completeness tests below.
            matched_rule = protected_rule
            detail = f"protected feature family ({protected_rule!r}); name rules not applied"
        elif identifier_rule is not None:
            reason = "identifier_like_column"
            matched_rule = identifier_rule
            detail = f"identifier / file / source-like name, matched {identifier_rule!r}"
        elif coordinate_rule is not None:
            reason = "raw_coordinate_column"
            matched_rule = coordinate_rule
            detail = f"raw screen coordinate, matched {coordinate_rule!r}"
        elif timestamp_rule is not None:
            reason = "timestamp_or_time_column"
            matched_rule = timestamp_rule
            detail = f"raw timestamp / time value, matched {timestamp_rule!r}"
        elif not coercion_ok or coercion_rate < MIN_COERCION_RATE:
            reason = "not_numeric_or_safely_coercible"
            detail = (
                f"coercion rate {coercion_rate:.3f} < MIN_COERCION_RATE {MIN_COERCION_RATE:.3f}"
                if coercion_ok
                else "column type cannot be converted to float"
            )
        elif n_unique_coerced <= 1:
            reason = "constant_column"
            detail = "a single distinct usable value carries no information"
        elif usable_rate < MIN_NON_MISSING_RATE:
            reason = "insufficient_non_missing_rate"
            detail = f"usable rate {usable_rate:.3f} < MIN_NON_MISSING_RATE {MIN_NON_MISSING_RATE:.3f}"

    row = {
        "column": column,
        "dtype": str(series.dtype),
        "is_numeric_dtype": is_numeric_dtype,
        "n_rows": n_rows,
        "n_non_null": n_non_null,
        "n_unique_raw": n_unique_raw,
        "n_unique_usable": n_unique_coerced,
        "n_infinite": n_infinite,
        "n_coercion_failures": n_coercion_failures,
        "coercion_rate": round(coercion_rate, 6),
        "usable_rate": round(usable_rate, 6),
        "protected_by_name_rules": protected,
        "status": "candidate" if reason is None else "excluded",
        "exclusion_reason": reason if reason is not None else "",
        "matched_rule": matched_rule if matched_rule is not None else "",
        "exclusion_detail": detail,
    }
    inventory_rows.append(row)

    if reason is None:
        candidate_features.append(column)
    else:
        excluded_rows.append(row)

feature_inventory = pd.DataFrame(inventory_rows)
excluded_features = pd.DataFrame(
    [r for r in excluded_rows],
    columns=list(feature_inventory.columns),
)

print(f"Columns in merged frame : {len(feature_inventory)}")
print(f"Candidate features      : {len(candidate_features)}")
print(f"Excluded columns        : {len(excluded_features)}")
print()
if len(excluded_features):
    print("Exclusion reasons:")
    print(excluded_features["exclusion_reason"].value_counts().to_string())
    print()
print("Candidate features:")
for name in candidate_features:
    flag = " (protected family)" if bool(
        feature_inventory.loc[feature_inventory["column"] == name, "protected_by_name_rules"].iloc[0]
    ) else ""
    print(f"  - {name}{flag}")


Columns in merged frame : 70
Candidate features      : 57
Excluded columns        : 13

Exclusion reasons:
exclusion_reason
raw_coordinate_column             6
timestamp_or_time_column          3
config_excluded_column            2
target_label_column               1
source_file_bookkeeping_column    1

Candidate features:
  - d0 (protected family)
  - d1 (protected family)
  - d2 (protected family)
  - TV0 (protected family)
  - TV1 (protected family)
  - TV2 (protected family)
  - HV0 (protected family)
  - HV1 (protected family)
  - HV2 (protected family)
  - VV0 (protected family)
  - VV1 (protected family)
  - VV2 (protected family)
  - A0 (protected family)
  - A1 (protected family)
  - A2 (protected family)
  - AM0 (protected family)
  - AM1 (protected family)
  - AM2 (protected family)
  - AV0 (protected family)
  - AV1 (protected family)
  - AV2 (protected family)
  - J0 (protected family)
  - J1 (protected family)
  - J2 (protected family)
  - avg_x (protected family)
  - avg

In [13]:
# =============================================================================
#  STEP 4 — MODEL REGISTRY
#  The three models of the project are fixed here. No search, no tuning.
#  The SVC's StandardScaler sits inside a Pipeline, so it can only ever be
#  fitted on the training portion of the split.
# =============================================================================
MODEL_DESCRIPTIONS = {
    "DummyClassifier_most_frequent": "Majority-class baseline; the floor every other model must clear.",
    "RBF_SVC_scaled": "RBF support vector classifier; StandardScaler fitted on training data only.",
    "RandomForest_balanced": "Random forest with balanced class weights; tree-based comparison.",
}


def build_models():
    """Return a fresh estimator per model so that no state is shared between features."""
    return {
        "DummyClassifier_most_frequent": DummyClassifier(strategy="most_frequent"),
        "RBF_SVC_scaled": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "svc",
                    SVC(
                        C=1.0,
                        gamma="scale",
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            class_weight="balanced",
        ),
    }


for name, description in MODEL_DESCRIPTIONS.items():
    print(f"{name:34s} : {description}")
print()
print("Hyper-parameters are fixed by the protocol. No grid search and no test-set selection.")


DummyClassifier_most_frequent      : Majority-class baseline; the floor every other model must clear.
RBF_SVC_scaled                     : RBF support vector classifier; StandardScaler fitted on training data only.
RandomForest_balanced              : Random forest with balanced class weights; tree-based comparison.

Hyper-parameters are fixed by the protocol. No grid search and no test-set selection.


In [14]:
# =============================================================================
#  STEP 5 — COMMON COHORT AND SINGLE-FEATURE SCREENING
# -----------------------------------------------------------------------------
#  PRIMARY MODE ("complete_case_all_candidates", the default):
#    1. one global cohort is built: label present, label class supported by at
#       least MIN_CLASS_COUNT rows, and EVERY candidate column finite after
#       numeric coercion (missing and +/-inf count as missing);
#    2. ONE stratified train/test split is drawn on that cohort;
#    3. every candidate feature is then evaluated on exactly the same training
#       rows, the same test rows and the same class space.
#    This is what makes the ranking between features meaningful. Screening each
#    feature on its own rows would produce different samples per feature and
#    could not be compared, even with an identical random_state.
#
#  EXPLORATORY MODE ("per_feature"): rebuilds a sample and a split per feature.
#    It is NOT rankable and must not be used for stage-2 selection.
#
#  The StandardScaler of the SVC lives inside a Pipeline: it is fitted on
#  training rows only. No tuning, and the test portion never selects anything.
# =============================================================================
SCREENING_COHORT_MODES = {
    "complete_case_all_candidates": (
        "Primary. One global complete-case cohort over all candidates and one single "
        "stratified split; every candidate is compared on the same rows and classes."
    ),
    "per_feature": (
        "Exploratory only. Rebuilds a sample and a split per feature, so the results "
        "are NOT rankable and must not be used for stage-2 selection."
    ),
}
if SCREENING_COHORT_MODE not in SCREENING_COHORT_MODES:
    raise ValueError(
        f"SCREENING_COHORT_MODE must be one of {sorted(SCREENING_COHORT_MODES)}"
    )

COHORT_MODE_IS_PRIMARY = SCREENING_COHORT_MODE == "complete_case_all_candidates"
MODE_VALIDITY = (
    RESULTS_VALIDITY
    if COHORT_MODE_IS_PRIMARY
    else "exploratory / not rankable - only valid after a real Colab run"
)

metrics_rows = []
failure_rows = []
screening_started_at = utc_now_iso()
screening_start = time.time()

cohort = None
cohort_error = ""
split_fingerprint = ""
n_rows_before_cohort = int(len(df))
n_rows_common_cohort = 0
n_train_common = 0
n_test_common = 0
n_classes_common = 0
train_pos = test_pos = None
cohort_accounting = {}


def numeric_finite_mask(series):
    """Numeric coercion of a column; returns (values, finite mask).

    Missing values, non-numeric values and +/-inf are all treated as not finite.
    """
    try:
        coerced = pd.to_numeric(series, errors="coerce")
        values = coerced.to_numpy(dtype="float64", na_value=np.nan)
    except (TypeError, ValueError):
        values = np.full(len(series), np.nan, dtype="float64")
    return values, np.isfinite(values)


def evaluate_on_split(feature, X_train, y_train, X_test, y_test, common_fields):
    """Fit the three models on one split. Returns (metric rows, failure rows)."""
    metric_rows = []
    failure_rows = []
    for model_name, estimator in build_models().items():
        model_start = time.time()
        try:
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_test)
        except Exception as exc:
            failure_rows.append(
                {
                    "feature": feature,
                    "model": model_name,
                    "stage": "model",
                    "reason": "model_fit_or_predict_failed",
                    "message": f"{type(exc).__name__}: {exc}",
                    "n_rows_used": int(len(X_train) + len(X_test)),
                    "timestamp_utc": utc_now_iso(),
                }
            )
            continue

        metric_rows.append(
            {
                "feature": feature,
                "model": model_name,
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "macro_precision": float(
                    precision_score(y_test, y_pred, average="macro", zero_division=0)
                ),
                "macro_recall": float(
                    recall_score(y_test, y_pred, average="macro", zero_division=0)
                ),
                "macro_f1": float(
                    f1_score(y_test, y_pred, average="macro", zero_division=0)
                ),
                "n_train": int(len(X_train)),
                "n_test": int(len(X_test)),
                "n_classes_total": int(common_fields["n_classes_common"]),
                "n_classes_used": int(common_fields["n_classes_common"]),
                "n_train_common": int(common_fields["n_train_common"]),
                "n_test_common": int(common_fields["n_test_common"]),
                "n_classes_common": int(common_fields["n_classes_common"]),
                "n_rows_before_cohort": int(common_fields["n_rows_before_cohort"]),
                "n_rows_common_cohort": int(common_fields["n_rows_common_cohort"]),
                "split_fingerprint": common_fields["split_fingerprint"],
                "n_non_null_full": int(common_fields["n_non_null_full"]),
                "n_infinite_full": int(common_fields["n_infinite_full"]),
                "n_coercion_failures_full": int(common_fields["n_coercion_failures_full"]),
                "n_unique_full": int(common_fields["n_unique_full"]),
                "n_unique_in_cohort": int(common_fields["n_unique_in_cohort"]),
                "feature_note": common_fields["feature_note"],
                "fit_seconds": round(time.time() - model_start, 4),
            }
        )
    return metric_rows, failure_rows


# -----------------------------------------------------------------------------
#  PRIMARY PATH: build the common cohort and draw the single global split
# -----------------------------------------------------------------------------
if COHORT_MODE_IS_PRIMARY:
    try:
        label_ok = df[LABEL_COLUMN].notna()
        class_counts_all = df.loc[label_ok, LABEL_COLUMN].value_counts()
        keep_classes = class_counts_all[class_counts_all >= MIN_CLASS_COUNT].index
        class_ok = df[LABEL_COLUMN].isin(keep_classes)

        complete_case = pd.Series(True, index=df.index)
        per_column_invalid = {}
        for feature in candidate_features:
            _, finite = numeric_finite_mask(df[feature])
            complete_case &= finite
            per_column_invalid[feature] = int((~finite).sum())

        cohort_mask = label_ok & class_ok & complete_case
        cohort = df.loc[cohort_mask].reset_index(drop=True)
        n_rows_common_cohort = int(len(cohort))

        cohort_accounting = {
            "n_rows_before_cohort": int(len(df)),
            "n_rows_dropped_label_missing": int((~label_ok).sum()),
            "n_rows_dropped_class_below_min_count": int((label_ok & ~class_ok).sum()),
            "n_rows_dropped_incomplete_candidates": int(
                (label_ok & class_ok & ~complete_case).sum()
            ),
            "n_rows_common_cohort": n_rows_common_cohort,
            "n_classes_seen": int(len(class_counts_all)),
            "n_classes_kept": int(len(keep_classes)),
            "n_classes_dropped_low_support": int(len(class_counts_all) - len(keep_classes)),
        }

        if n_rows_common_cohort < MIN_ROWS_PER_FEATURE:
            raise RuntimeError(
                f"the common cohort holds {n_rows_common_cohort} rows, which is below "
                f"MIN_ROWS_PER_FEATURE ({MIN_ROWS_PER_FEATURE}); no ranking is produced"
            )
        if cohort[LABEL_COLUMN].nunique() < 2:
            raise RuntimeError(
                "the common cohort holds fewer than two labels; no ranking is produced"
            )

        y_cohort = cohort[LABEL_COLUMN].to_numpy()
        positions = np.arange(n_rows_common_cohort)
        try:
            train_pos, test_pos = train_test_split(
                positions,
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=y_cohort,
            )
        except Exception as exc:
            raise RuntimeError(
                "the common cohort cannot be split in a stratified way "
                f"({type(exc).__name__}: {exc}); no ranking is produced"
            ) from exc

        train_classes = set(np.unique(y_cohort[train_pos]))
        test_classes = set(np.unique(y_cohort[test_pos]))
        if train_classes != test_classes:
            raise RuntimeError(
                "the stratified split did not preserve the class space across the two "
                "portions; no ranking is produced"
            )

        n_train_common = int(len(train_pos))
        n_test_common = int(len(test_pos))
        n_classes_common = int(len(train_classes))

        fingerprint_payload = json.dumps(
            {
                "n_rows_common_cohort": n_rows_common_cohort,
                "test_size": TEST_SIZE,
                "random_state": RANDOM_STATE,
                "train": sorted(int(i) for i in train_pos),
                "test": sorted(int(i) for i in test_pos),
            },
            sort_keys=True,
        )
        split_fingerprint = "sha256:" + sha256_text(fingerprint_payload)[:32]

    except Exception as exc:
        cohort = None
        train_pos = test_pos = None
        cohort_error = f"{type(exc).__name__}: {exc}"
        failure_rows.append(
            {
                "feature": "",
                "model": "",
                "stage": "cohort",
                "reason": "common_cohort_failed",
                "message": cohort_error,
                "n_rows_used": 0,
                "timestamp_utc": utc_now_iso(),
            }
        )
        print("!" * 78)
        print("COMMON COHORT FAILED - NO PRIMARY RANKING IS PRODUCED")
        print(cohort_error)
        print("!" * 78)

# -----------------------------------------------------------------------------
#  SCREENING
# -----------------------------------------------------------------------------
if COHORT_MODE_IS_PRIMARY and cohort is not None:
    common_fields = {
        "n_rows_before_cohort": n_rows_before_cohort,
        "n_rows_common_cohort": n_rows_common_cohort,
        "n_train_common": n_train_common,
        "n_test_common": n_test_common,
        "n_classes_common": n_classes_common,
        "split_fingerprint": split_fingerprint,
    }

    for feature in candidate_features:
        feature_start = time.time()
        values, finite = numeric_finite_mask(df[feature])
        column_diagnostics = {
            "n_non_null_full": int(df[feature].notna().sum()),
            "n_infinite_full": int(
                np.isinf(
                    pd.to_numeric(df[feature], errors="coerce")
                    .to_numpy(dtype="float64", na_value=np.nan)
                ).sum()
            ),
            "n_coercion_failures_full": int(
                df[feature].notna().sum() - pd.to_numeric(df[feature], errors="coerce").notna().sum()
            ),
            "n_unique_full": int(df[feature].nunique(dropna=True)),
        }

        # the cohort is already complete-case, so every value here is finite
        cohort_values = values[cohort_mask.to_numpy()]
        X = cohort_values.reshape(-1, 1)
        y_all = cohort[LABEL_COLUMN].to_numpy()

        n_unique_in_cohort = int(len(np.unique(cohort_values)))
        feature_note = (
            "constant_within_common_cohort"
            if n_unique_in_cohort <= 1
            else ""
        )

        fields = dict(common_fields)
        fields.update(column_diagnostics)
        fields["n_unique_in_cohort"] = n_unique_in_cohort
        fields["feature_note"] = feature_note

        rows, fails = evaluate_on_split(
            feature,
            X[train_pos],
            y_all[train_pos],
            X[test_pos],
            y_all[test_pos],
            fields,
        )
        metrics_rows.extend(rows)
        failure_rows.extend(fails)

        print(
            f"[screened] {feature:<24s} cohort_rows={n_rows_common_cohort:>6d} "
            f"train={n_train_common:>6d} test={n_test_common:>5d} "
            f"classes={n_classes_common:>3d}  ({time.time() - feature_start:.2f}s)"
        )

else:
    # -------------------------------------------------------------------------
    #  EXPLORATORY PATH ONLY: a separate sample and split per feature.
    #  The results are not comparable between features and are not rankable.
    # -------------------------------------------------------------------------
    print("EXPLORATORY MODE: per-feature samples and splits. NOT rankable.")
    for feature in candidate_features:
        feature_start = time.time()
        values, finite = numeric_finite_mask(df[feature])
        label_ok = df[LABEL_COLUMN].notna()
        usable = finite & label_ok.to_numpy()
        if int(usable.sum()) < MIN_ROWS_PER_FEATURE:
            failure_rows.append(
                {
                    "feature": feature,
                    "model": "",
                    "stage": "screening",
                    "reason": "insufficient_usable_rows",
                    "message": (
                        f"{int(usable.sum())} usable rows < MIN_ROWS_PER_FEATURE "
                        f"{MIN_ROWS_PER_FEATURE}"
                    ),
                    "n_rows_used": int(usable.sum()),
                    "timestamp_utc": utc_now_iso(),
                }
            )
            continue

        y_series = df.loc[usable, LABEL_COLUMN]
        counts = y_series.value_counts()
        kept = counts[counts >= MIN_CLASS_COUNT].index
        if len(kept) < 2:
            failure_rows.append(
                {
                    "feature": feature,
                    "model": "",
                    "stage": "screening",
                    "reason": "insufficient_classes_after_min_class_count",
                    "message": f"{len(kept)} class(es) with at least {MIN_CLASS_COUNT} rows",
                    "n_rows_used": int(usable.sum()),
                    "timestamp_utc": utc_now_iso(),
                }
            )
            continue

        keep = usable & df[LABEL_COLUMN].isin(kept).to_numpy()
        X = values[keep].reshape(-1, 1)
        y_all = df.loc[keep, LABEL_COLUMN].to_numpy()
        try:
            tr, te = train_test_split(
                np.arange(len(X)),
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=y_all,
            )
        except Exception as exc:
            failure_rows.append(
                {
                    "feature": feature,
                    "model": "",
                    "stage": "split",
                    "reason": "split_failed",
                    "message": f"{type(exc).__name__}: {exc}",
                    "n_rows_used": int(len(X)),
                    "timestamp_utc": utc_now_iso(),
                }
            )
            continue

        fields = {
            "n_rows_before_cohort": int(len(df)),
            "n_rows_common_cohort": 0,
            "n_train_common": 0,
            "n_test_common": 0,
            "n_classes_common": int(len(kept)),
            "split_fingerprint": "",
            "n_non_null_full": int(df[feature].notna().sum()),
            "n_infinite_full": int(
                np.isinf(
                    pd.to_numeric(df[feature], errors="coerce")
                    .to_numpy(dtype="float64", na_value=np.nan)
                ).sum()
            ),
            "n_coercion_failures_full": int(
                df[feature].notna().sum()
                - pd.to_numeric(df[feature], errors="coerce").notna().sum()
            ),
            "n_unique_full": int(df[feature].nunique(dropna=True)),
            "n_unique_in_cohort": int(len(np.unique(values[keep]))),
            "feature_note": "exploratory_per_feature_sample_not_rankable",
        }
        rows, fails = evaluate_on_split(feature, X[tr], y_all[tr], X[te], y_all[te], fields)
        metrics_rows.extend(rows)
        failure_rows.extend(fails)

        print(
            f"[exploratory] {feature:<22s} rows={len(X):>6d} "
            f"train={len(tr):>6d} test={len(te):>5d}  ({time.time() - feature_start:.2f}s)"
        )

screening_seconds = time.time() - screening_start

single_feature_metrics = pd.DataFrame(
    metrics_rows,
    columns=[
        "feature",
        "model",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "cohort_mode",
        "n_train",
        "n_test",
        "n_classes_total",
        "n_classes_used",
        "n_train_common",
        "n_test_common",
        "n_classes_common",
        "n_rows_before_cohort",
        "n_rows_common_cohort",
        "split_fingerprint",
        "n_non_null_full",
        "n_infinite_full",
        "n_coercion_failures_full",
        "n_unique_full",
        "n_unique_in_cohort",
        "feature_note",
        "fit_seconds",
    ],
)
single_feature_failures = pd.DataFrame(
    failure_rows,
    columns=[
        "feature",
        "model",
        "stage",
        "reason",
        "message",
        "n_rows_used",
        "timestamp_utc",
    ],
)

if not single_feature_metrics.empty:
    single_feature_metrics["cohort_mode"] = SCREENING_COHORT_MODE

print()
print(f"Cohort mode                : {SCREENING_COHORT_MODE}")
print(f"Candidate features screened : {len(candidate_features)}")
print(f"Metric rows written         : {len(single_feature_metrics)}")
print(f"Failures recorded           : {len(single_feature_failures)}")
if cohort_accounting:
    for key, value in cohort_accounting.items():
        print(f"  {key:<38s}: {value}")
if split_fingerprint:
    print(f"  {'split_fingerprint':<38s}: {split_fingerprint}")
print(f"Screening wall time          : {screening_seconds:.1f} s")
if len(single_feature_failures):
    print()
    print("Failure reasons:")
    print(single_feature_failures["reason"].value_counts().to_string())


[screened] d0                       cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.71s)
[screened] d1                       cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.69s)
[screened] d2                       cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.69s)
[screened] TV0                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (1.09s)
[screened] TV1                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (1.06s)
[screened] TV2                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (1.06s)
[screened] HV0                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.92s)
[screened] HV1                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.89s)
[screened] HV2                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.88s)
[screened] VV0                      cohort_rows=  2002 train=  1601 test=  401 classes=  9  (0.82s)


In [15]:
# =============================================================================
#  STEP 6 — WRITE THE OUTPUTS
#  Five files, all aggregated, all stamped with the validity notice.
#  Raw rows, raw values, participant file names, label values and local
#  absolute paths are never written.
# =============================================================================
output_dir = ensure_output_dir(OUTPUT_DIR)

inventory_path = write_table(feature_inventory, output_dir / "feature_inventory.csv")
excluded_path = write_table(excluded_features, output_dir / "excluded_features.csv")
metrics_path = write_table(
    single_feature_metrics, output_dir / "single_feature_metrics.csv", validity=MODE_VALIDITY
)
failures_path = write_table(
    single_feature_failures, output_dir / "single_feature_failures.csv", validity=MODE_VALIDITY
)

file_digest = sha256_text("\n".join(p.name for p in sorted(source_files)))
label_counts_series = df[LABEL_COLUMN].value_counts(dropna=True)

run_status = "completed"
if cohort_error:
    run_status = "failed - no primary ranking produced"
elif not COHORT_MODE_IS_PRIMARY:
    run_status = "completed in exploratory mode - results are not rankable"

run_metadata = {
    # --- the fields the run contract requires, at the top level --------------
    "results_validity": MODE_VALIDITY,
    "cohort_mode": SCREENING_COHORT_MODE,
    "cohort_mode_description": SCREENING_COHORT_MODES[SCREENING_COHORT_MODE],
    "n_rows_before_cohort": int(n_rows_before_cohort),
    "n_rows_common_cohort": int(n_rows_common_cohort),
    "n_train_common": int(n_train_common),
    "n_test_common": int(n_test_common),
    "n_classes_common": int(n_classes_common),
    "split_fingerprint": split_fingerprint,
    # --- run context ---------------------------------------------------------
    "notice": (
        "Every number in these outputs is produced only by a real Colab run on the real "
        "course data. Any value taken from this repository before such a run is not a result."
    ),
    "stage": "V3 stage 1 - all-candidate single-feature screening",
    "run_status": run_status,
    "run_timestamp_utc": utc_now_iso(),
    "screening_started_utc": screening_started_at,
    "screening_seconds": round(screening_seconds, 2),
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "configuration": {
        "DATA_DIR": "<withheld: absolute local path is not recorded>",
        "FILE_PATTERN": FILE_PATTERN,
        "LABEL_COLUMN": LABEL_COLUMN,
        "RANDOM_STATE": RANDOM_STATE,
        "TEST_SIZE": TEST_SIZE,
        "SCREENING_COHORT_MODE": SCREENING_COHORT_MODE,
        "MIN_NON_MISSING_RATE": MIN_NON_MISSING_RATE,
        "MIN_CLASS_COUNT": MIN_CLASS_COUNT,
        "MIN_ROWS_PER_FEATURE": MIN_ROWS_PER_FEATURE,
        "MIN_COERCION_RATE": MIN_COERCION_RATE,
        "OUTPUT_DIR": "<withheld: absolute local path is not recorded>",
        "EXCLUDED_COLUMNS": list(EXCLUDED_COLUMNS),
        "EXCLUDED_COLUMN_PATTERNS": list(EXCLUDED_COLUMN_PATTERNS),
        "PROTECTED_COLUMN_PATTERNS": list(PROTECTED_COLUMN_PATTERNS),
        "IDENTIFIER_COLUMN_PATTERNS": list(IDENTIFIER_COLUMN_PATTERNS),
        "RAW_COORDINATE_PATTERNS": list(RAW_COORDINATE_PATTERNS),
        "TIMESTAMP_PATTERNS": list(TIMESTAMP_PATTERNS),
        "ALLOW_OUTPUT_IN_GIT_REPO": ALLOW_OUTPUT_IN_GIT_REPO,
        "INCLUDE_SOURCE_FILE_NAMES": INCLUDE_SOURCE_FILE_NAMES,
        "INCLUDE_LABEL_VALUES": INCLUDE_LABEL_VALUES,
    },
    "cohort": {
        "definition": (
            "Rows whose label is present, whose label class has at least MIN_CLASS_COUNT rows, "
            "and for which every candidate column is finite after numeric coercion "
            "(missing and +/-inf both count as missing)."
        ),
        "accounting": cohort_accounting,
        "error": cohort_error,
        "split": {
            "type": "one single stratified train_test_split on the common cohort",
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "shared_by_every_candidate_feature": bool(COHORT_MODE_IS_PRIMARY),
        },
    },
    "data_footprint": {
        "n_files_matched": len(source_files),
        "n_files_loaded": len(frames),
        "n_files_skipped": len(skipped_files),
        "source_file_set_sha256": file_digest,
        "n_rows_merged": int(df.shape[0]),
        "n_columns_merged": int(df.shape[1]),
        "n_label_classes": int(len(label_counts_series)),
        "label_class_count_min": int(label_counts_series.min()) if len(label_counts_series) else 0,
        "label_class_count_median": (
            int(np.median(label_counts_series.to_numpy())) if len(label_counts_series) else 0
        ),
        "label_class_count_max": int(label_counts_series.max()) if len(label_counts_series) else 0,
        "label_class_counts": (
            {str(k): int(v) for k, v in label_counts_series.items()}
            if INCLUDE_LABEL_VALUES
            else "withheld (INCLUDE_LABEL_VALUES=False)"
        ),
        "source_files": (
            [r["file"] for r in file_records]
            if INCLUDE_SOURCE_FILE_NAMES
            else "withheld (INCLUDE_SOURCE_FILE_NAMES=False)"
        ),
        "skipped_files": (
            skipped_files if INCLUDE_SOURCE_FILE_NAMES else [r["reason"] for r in skipped_files]
        ),
    },
    "feature_pool": {
        "n_columns": int(len(feature_inventory)),
        "n_candidates": len(candidate_features),
        "candidate_features": list(candidate_features),
        "n_excluded": int(len(excluded_features)),
        "exclusion_reason_counts": (
            excluded_features["exclusion_reason"].value_counts().to_dict()
            if len(excluded_features)
            else {}
        ),
    },
    "screening": {
        "models": MODEL_DESCRIPTIONS,
        "model_hyperparameters": {
            "DummyClassifier_most_frequent": {"strategy": "most_frequent"},
            "RBF_SVC_scaled": {
                "C": 1.0,
                "gamma": "scale",
                "class_weight": "balanced",
                "scaler": "StandardScaler fitted on the training portion only (Pipeline)",
            },
            "RandomForest_balanced": {
                "n_estimators": 200,
                "random_state": RANDOM_STATE,
                "class_weight": "balanced",
            },
        },
        "n_metric_rows": int(len(single_feature_metrics)),
        "n_failure_rows": int(len(single_feature_failures)),
        "failure_reason_counts": (
            single_feature_failures["reason"].value_counts().to_dict()
            if len(single_feature_failures)
            else {}
        ),
    },
    "outputs": {
        "feature_inventory.csv": sha256_text(inventory_path.read_text(encoding="utf-8")),
        "excluded_features.csv": sha256_text(excluded_path.read_text(encoding="utf-8")),
        "single_feature_metrics.csv": sha256_text(metrics_path.read_text(encoding="utf-8")),
        "single_feature_failures.csv": sha256_text(failures_path.read_text(encoding="utf-8")),
    },
    "protocol_notes": [
        "The label is read from inside the CSV files only; it is never derived from a file name.",
        "Single-participant CSV files are merged before the class space is inspected, because a single file usually carries a single label.",
        "In the primary mode every candidate feature is evaluated on the same cohort rows, the same training rows, the same test rows and the same class space.",
        "A stratified split with an identical seed on different row sets is not the same split, so per-feature samples are never used for the primary ranking.",
        "The StandardScaler of the SVC is fitted on training data only.",
        "No hyper-parameter search was run and the test portion was never used for selection.",
        "A feature or model that fails is recorded in single_feature_failures.csv and the run continues.",
        "Raw data, participant file names, label values and local absolute paths are never written.",
    ],
}

if not COHORT_MODE_IS_PRIMARY:
    run_metadata["warning"] = (
        "This run used the exploratory per_feature mode. The metrics are NOT comparable "
        "between features, are NOT rankable and must NOT be used for stage-2 selection. "
        "Re-run with SCREENING_COHORT_MODE = 'complete_case_all_candidates'."
    )

metadata_path = dump_json(output_dir / "run_metadata.json", run_metadata)

print(f"Output directory: {output_dir}")
for path in [inventory_path, excluded_path, metrics_path, failures_path, metadata_path]:
    print(f"  written: {path.name}  ({path.stat().st_size} bytes)")
print()
print(f"cohort mode      : {SCREENING_COHORT_MODE}")
print(f"rows before      : {n_rows_before_cohort}")
print(f"common cohort    : {n_rows_common_cohort} rows")
print(f"train / test     : {n_train_common} / {n_test_common}")
print(f"classes (common) : {n_classes_common}")
print(f"split fingerprint: {split_fingerprint or '(none)'}")
print(f"run status       : {run_status}")
print()
print("These files are aggregated and contain no raw rows, no participant values and no")
print("local absolute paths. They are valid only as the record of a real Colab run.")


Output directory: /content/outputs/v3_single_feature
  written: feature_inventory.csv  (14725 bytes)
  written: excluded_features.csv  (3045 bytes)
  written: single_feature_metrics.csv  (46572 bytes)
  written: single_feature_failures.csv  (78 bytes)
  written: run_metadata.json  (7450 bytes)

cohort mode      : complete_case_all_candidates
rows before      : 2002
common cohort    : 2002 rows
train / test     : 1601 / 401
classes (common) : 9
split fingerprint: sha256:297284c375de6cb3535fbd2268907915
run status       : completed

These files are aggregated and contain no raw rows, no participant values and no
local absolute paths. They are valid only as the record of a real Colab run.


In [16]:
# =============================================================================
#  STEP 7 — RUN SUMMARY
#  Prints what was produced. Every number below comes from this run; nothing
#  is pre-filled. With no executed run, this cell has never printed anything.
# =============================================================================
print("=" * 78)
print("V3 STAGE 1 - SINGLE-FEATURE SCREENING SUMMARY")
print(f"run timestamp (UTC): {utc_now_iso()}")
print("=" * 78)
print()

print("Run footprint")
print(f"  cohort mode                   : {SCREENING_COHORT_MODE}")
print(f"  files matched / loaded / skip : {len(source_files)} / {len(frames)} / {len(skipped_files)}")
print(f"  rows merged                   : {df.shape[0]}")
print(f"  candidate features            : {len(candidate_features)}")
print(f"  excluded columns              : {len(excluded_features)}")
print(f"  metric rows                   : {len(single_feature_metrics)}")
print(f"  failure rows                  : {len(single_feature_failures)}")
print()

if cohort_accounting:
    print("Common cohort")
    for key, value in cohort_accounting.items():
        print(f"  {key:<38s}: {value}")
    print(f"  {'split_fingerprint':<38s}: {split_fingerprint or '(none)'}")
    print()

if len(single_feature_metrics):
    print("Shared evaluation split (must be identical for every candidate)")
    shared = single_feature_metrics[
        ["n_train_common", "n_test_common", "n_classes_common", "split_fingerprint"]
    ].drop_duplicates()
    print(shared.to_string(index=False))
    consistent = len(shared) == 1
    print(f"  identical across all candidates: {consistent}")
    print()

    print("Best features per model, ranked by macro F1 (held-out portion)")
    for model_name in MODEL_DESCRIPTIONS:
        subset = single_feature_metrics[single_feature_metrics["model"] == model_name]
        if not len(subset):
            continue
        ranked = subset.sort_values("macro_f1", ascending=False).head(5)
        print()
        print(f"  {model_name}")
        print(
            ranked[
                [
                    "feature",
                    "accuracy",
                    "macro_precision",
                    "macro_recall",
                    "macro_f1",
                    "n_train",
                    "n_test",
                    "n_classes_common",
                ]
            ].to_string(index=False)
        )

    dummy = single_feature_metrics[
        single_feature_metrics["model"] == "DummyClassifier_most_frequent"
    ]
    if len(dummy):
        print()
        print(
            "  Majority-class floor (median across features): "
            f"accuracy {dummy['accuracy'].median():.4f} | "
            f"macro F1 {dummy['macro_f1'].median():.4f}"
        )
        print("  A feature is only interesting if it clears this floor on the same split.")

if len(single_feature_failures):
    print()
    print("Recorded failures")
    print(single_feature_failures["reason"].value_counts().to_string())

print()
if not COHORT_MODE_IS_PRIMARY:
    print("WARNING: this run used the exploratory per_feature mode.")
    print("The rows above are NOT comparable between features and are NOT rankable.")
    print("Re-run with SCREENING_COHORT_MODE = 'complete_case_all_candidates'.")
print("Reminder: these numbers describe this run only. They are not valid until the")
print("notebook has been executed in Colab on the real course data.")


V3 STAGE 1 - SINGLE-FEATURE SCREENING SUMMARY
run timestamp (UTC): 2026-11-13T10:11:17+00:00

Run footprint
  cohort mode                   : complete_case_all_candidates
  files matched / loaded / skip : 10 / 10 / 0
  rows merged                   : 2002
  candidate features            : 57
  excluded columns              : 13
  metric rows                   : 171
  failure rows                  : 0

Common cohort
  n_rows_before_cohort                  : 2002
  n_rows_dropped_label_missing          : 0
  n_rows_dropped_class_below_min_count  : 0
  n_rows_dropped_incomplete_candidates  : 0
  n_rows_common_cohort                  : 2002
  n_classes_seen                        : 9
  n_classes_kept                        : 9
  n_classes_dropped_low_support         : 0
  split_fingerprint                     : sha256:297284c375de6cb3535fbd2268907915

Shared evaluation split (must be identical for every candidate)
 n_train_common  n_test_common  n_classes_common                       split

## Output reference

All five files are written to `OUTPUT_DIR` and carry the column `results_validity` with the value
`only valid after a real Colab run` (the exploratory mode carries
`exploratory / not rankable - only valid after a real Colab run` instead). The manifest carries the
same statement as a JSON field. No file contains a raw data row, a participant value or a local
absolute path.

### `feature_inventory.csv`

One row per column of the merged frame.

| Column | Meaning |
| --- | --- |
| `column` | Column name |
| `dtype` | pandas dtype of the merged column |
| `is_numeric_dtype` | Whether pandas typed the column as numeric |
| `n_rows` | Rows in the merged frame |
| `n_non_null` | Non-null values |
| `n_unique_raw` | Distinct values before coercion |
| `n_unique_usable` | Distinct values after numeric coercion |
| `n_infinite` | Infinite values found |
| `n_coercion_failures` | Non-null values that could not be converted to float |
| `coercion_rate` | Share of non-null values convertible to float |
| `usable_rate` | Share of rows with a finite numeric value |
| `protected_by_name_rules` | Whether the column belongs to a protected feature family |
| `status` | `candidate` or `excluded` |
| `exclusion_reason` | The rule that rejected the column (empty for candidates) |
| `matched_rule` | The pattern that triggered the rule |
| `exclusion_detail` | Human-readable explanation |

### `excluded_features.csv`

The same schema, restricted to the rejected columns. This table is the audit trail: a reader can
check that nothing was dropped silently, that the protected families were kept, and which rule was
responsible.

### `single_feature_metrics.csv`

One row per (candidate feature × model).

| Column | Meaning |
| --- | --- |
| `feature` | Candidate feature name |
| `model` | `DummyClassifier_most_frequent`, `RBF_SVC_scaled` or `RandomForest_balanced` |
| `accuracy` | Held-out accuracy |
| `macro_precision` | Macro-averaged precision |
| `macro_recall` | Macro-averaged recall |
| `macro_f1` | Macro-averaged F1 |
| `cohort_mode` | The mode the run used (`complete_case_all_candidates` in the primary protocol) |
| `n_train` / `n_test` | Size of the two split portions — **identical for every candidate feature** |
| `n_classes_total` / `n_classes_used` | Classes in the shared cohort |
| `n_train_common` / `n_test_common` | The shared split sizes, repeated on every row for verification |
| `n_classes_common` | The shared class count, repeated on every row for verification |
| `n_rows_before_cohort` | Rows in the merged frame before the cohort was built |
| `n_rows_common_cohort` | Rows in the common cohort |
| `split_fingerprint` | Irreversible digest of the single shared split |
| `n_non_null_full` | Non-null values of the column over the whole merged frame (diagnostic) |
| `n_infinite_full` | Infinite values of the column over the whole merged frame (diagnostic) |
| `n_coercion_failures_full` | Values that could not be converted, over the whole merged frame (diagnostic) |
| `n_unique_full` | Distinct values over the whole merged frame (diagnostic) |
| `n_unique_in_cohort` | Distinct values inside the common cohort (diagnostic) |
| `feature_note` | Empty, or `constant_within_common_cohort` when the feature has no variation in the cohort |
| `fit_seconds` | Fit and predict time for this row |

The `n_*_common` columns and the `split_fingerprint` are repeated on every row on purpose: a reader
can confirm with one glance that every candidate was measured on the same split.

### `single_feature_failures.csv`

One row per feature (or feature × model) that could not be evaluated.

| Column | Meaning |
| --- | --- |
| `feature` | Affected feature (empty for a cohort-level failure) |
| `model` | Affected model, empty for a feature-level failure |
| `stage` | `cohort`, `coercion`, `screening`, `split` or `model` |
| `reason` | Machine-readable reason code |
| `message` | Detail, including the exception text for model failures |
| `n_rows_used` | The quantity that was compared |
| `timestamp_utc` | When the failure was recorded |

### `run_metadata.json`

The manifest of the run. The fields the run contract requires are at the **top level**:

`results_validity`, `cohort_mode`, `cohort_mode_description`, `n_rows_before_cohort`,
`n_rows_common_cohort`, `n_train_common`, `n_test_common`, `n_classes_common`,
`split_fingerprint`, `run_status`, `run_timestamp_utc`.

Below them: the environment, the configuration snapshot (with `DATA_DIR` and `OUTPUT_DIR`
replaced by a placeholder, because a local absolute path must not be recorded), the cohort
definition and row accounting, the data footprint (counts and digests, not raw values), the
candidate pool, the model roster and hyper-parameters, the failure summary, a SHA-256 digest of
each written table, and the protocol notes.

---

## Reading the results honestly

* **Macro metrics are the headline.** The task is multi-class identification with unbalanced
  classes, so a micro-average would let the largest class dominate.
* **The dummy model is the floor, not a competitor.** Its accuracy is the majority-class share of
  the test portion; a feature that does not clear it has demonstrated nothing.
* **The ranking is only valid for the cohort it was measured on.** The common cohort is a
  complete-case sample, so it is smaller than the merged data; features that lose rows to the
  complete-case rule are disadvantaged relative to what they would achieve on their own rows. The
  cohort accounting in the manifest states how many rows were lost and why.
* **One feature at a time is a weak model.** A single-feature score is evidence about *information
  content*, not about achievable performance. Several weak features can combine into a strong
  model, and one strong feature can be an artefact of the split.
* **The three models are compared under one protocol** on the same rows, which is what makes the
  comparison meaningful.


## Limitations of this stage

1. **The complete-case cohort is smaller than the merged data.** A row survives only if *every*
   candidate column is finite. A feature that is incomplete in a few rows therefore costs those
   rows for *all* features. This is the price of a like-for-like comparison, and the manifest
   records how many rows were dropped and for which reason.
2. **The cohort is a biased subsample.** Rows with missing or infinite values are not missing at
   random: they are usually the harder or shorter movements. The ranking therefore describes the
   complete-case rows, not the whole dataset, and the complete-case rows may be easier to
   separate than the full data.
3. **Segment-level random split.** Rows are individual three-point segments, and neighbouring
   segments of one recording are not independent. A random split can place related segments on both
   sides, so the scores are optimistic as a generalisation estimate. A session-level split would be
   the stricter protocol, and no session identifier is available in the delivered data.
4. **The excluded columns are excluded on principle, not on measurement.** Raw coordinates
   (`x0`–`x2`, `y0`–`y2`), raw times (`t0`–`t2`), the blank-header index column and identifiers
   such as `Seg_id` are removed because they plausibly encode the recording rather than the
   participant. This is a defensible restriction, but it is a restriction: a model allowed to use
   them would score higher, and part of that gain would not survive a new session.
5. **Single-feature screening is not feature selection.** The ranking produced here is evidence for
   stage 2, not a decision. Stage 2 must evaluate candidate sets under the same protocol, and the
   best single feature is not automatically part of the best set.
6. **Small samples and few labels.** Macro metrics move when a single class is small. The failure
   table and the class counts in the manifest are the context in which any number must be read.
7. **One split, one seed.** The protocol uses a single stratified split with `RANDOM_STATE = 42`.
   That is reproducible, but it does not quantify split-to-split variation. If stage 2 selects a
   feature set, repeated splits or cross-validation on the training portion should support that
   choice.
8. **Behavioural biometric data.** The data describes how a person moves a mouse. It is sensitive
   material; the course and institutional handling requirements apply, and only aggregated,
   participant-free tables belong in the repository.
9. **Results are conditional.** Every number produced by this notebook is valid only for the run
   that produced it, on the dataset version that was loaded, with the configuration recorded in
   `run_metadata.json`.

## Integrity statement for this notebook

* It contains **no stored outputs** and **no result values**. It cannot be mistaken for evidence
  of an experiment that has not been run.
* It contains **no bundled data** and no download of the course data. It reads only the directory
  you configure, at run time.
* It **never derives a label from a file name**. A file without the label column is skipped and
  reported; the label space comes from the CSV content alone.
* It **merges all participant files before the class space is inspected**, because a single file
  usually carries a single label and the class space only exists across the merged data.
* It evaluates every candidate on **one common cohort and one shared stratified split**. It never
  rebuilds a sample per feature for the primary ranking, and it does not claim that an identical
  seed implies an identical split.
* It **never writes raw data**, never records a local absolute path, and it refuses to write its
  outputs into a Git repository.
* It performs **no hyper-parameter tuning** and never uses the test portion to select anything.
* The label values, the participant-linked file names and the raw rows are withheld from the
  outputs by default; the switches that would include them are in the configuration cell and are
  set to `False`.
* The only environment in which this notebook may be executed against the real course data is a
  Colab runtime (or an equivalent private environment) that is not committed to the repository.

---

## Stop point

This completes **stage 1 only**. The next step — evaluating candidate *feature sets* with the
same models and the same protocol — is a separate notebook and must not begin until this stage has
been run on the real data, its outputs reviewed, and the supervisor's approval given.

**Checklist before stage 2**

- [ ] The notebook has been executed in Colab against the real data.
- [ ] `run_metadata.json` exists and records the configuration, the cohort accounting and the
      split fingerprint of that run.
- [ ] `excluded_features.csv` has been reviewed: nothing important was excluded by accident, and
      the protected families (`avg_*`, `min_*`, `max_*`, `d0`–`d2`, `TV0`–`TV2`, `HV0`–`HV2`,
      `VV0`–`VV2`, `A0`–`A2`, `AM0`–`AM2`, `AV0`–`AV2`, `J0`–`J2`) are present as candidates.
- [ ] `single_feature_metrics.csv` shows one `split_fingerprint` and identical `n_train` /
      `n_test` / `n_classes_common` on every row.
- [ ] `single_feature_failures.csv` has been reviewed: no systematic failure is hiding.
- [ ] The majority-class floor from the dummy model is known and used as the reference.
- [ ] The supervisor has reviewed the stage-1 output.
